### Init db

In [1]:
import lancedb
import pandas as pd

In [2]:
db = lancedb.connect("./amazon_products")

In [3]:
products_df = pd.read_csv("amazon_beauty_products_1000.csv")

In [4]:
products_df.head()

,parent_asin,title
0,B01LZG89RR,Careline Super Long Lasting Automatic Eye Penc...
1,B09LLV9TPV,Dailinna Corn Wave Ponytail Extension Clip in ...
2,B07J4VNYW1,"ZUM Holiday Mint Blitzum Mist, 4 FZ"
3,B08KFFB2PP,Perfectly Posh PLAY DATE Super Moisturizing Nu...
4,B09MQ53X93,New for 2021 Bath and Body Works Gingham Heart...


### compute embeddings

In [5]:
from lancedb.embeddings import get_registry

In [6]:
embedding_func = get_registry().get("openai").create(name="text-embedding-3-small")

In [7]:
emb = embedding_func.compute_query_embeddings("O Naturals bar of soap")

In [8]:
type(emb), len(emb), len(emb[0])

(list, 1, 1536)

In [9]:
emb[0][:5]

[0.02231518365442753,
 0.02703782729804516,
 -0.034908901900053024,
 0.020644305273890495,
 -0.00884460099041462]

### create table

In [10]:
from lancedb.pydantic import LanceModel, Vector

In [11]:
class Product(LanceModel):
    parent_asin: str
    title: str = embedding_func.SourceField()
    vector: Vector(embedding_func.ndims()) = embedding_func.VectorField()

In [12]:
table = db.create_table("amazon_products", products_df, schema=Product)

In [13]:
db.table_names()

['amazon_products']

In [14]:
table.to_pandas().head()

,parent_asin,title,vector
0,B01LZG89RR,Careline Super Long Lasting Automatic Eye Penc...,"[-0.007478258, -0.0006299982, 0.013862463, 0.0..."
1,B09LLV9TPV,Dailinna Corn Wave Ponytail Extension Clip in ...,"[0.012858504, 0.012725714, -0.01755042, 0.0348..."
2,B07J4VNYW1,"ZUM Holiday Mint Blitzum Mist, 4 FZ","[-0.0106983, 0.0218014, -0.026095178, 0.003532..."
3,B08KFFB2PP,Perfectly Posh PLAY DATE Super Moisturizing Nu...,"[-0.030492116, -0.029538527, -0.06270977, 0.00..."
4,B09MQ53X93,New for 2021 Bath and Body Works Gingham Heart...,"[-0.019150602, 0.02508645, -0.010024986, 0.033..."


### Sample vector search

In [17]:
test_query = "mens perfume"
test_results = table.search(test_query, query_type="vector").limit(5)

In [22]:
test_results.to_pandas()

,parent_asin,title,vector,_distance
0,B09HL8SVMJ,Michael Malul OCEAN NOIR 3.4 EAU DE PARFUM NEW...,"[0.030007359, 0.02743247, -0.037385404, -0.022...",0.881282
1,B00D7EWILA,"Hermes Terre D'hermes Pure Perfume for Men, Li...","[0.02430687, 0.032649007, -0.037101142, -0.013...",0.886635
2,B01HQFOWOQ,Kenzo Jeu d'Amour l'Elixir Eau de Parfum Inten...,"[0.001010046, 0.03055501, -0.07820173, -0.0382...",0.911542
3,B000SSN66Y,Gianni Versace Dreamer Men's 3.3-ounce Eau de ...,"[0.036253627, 0.004074725, -0.05582168, -0.066...",0.924948
4,B08P5JFGF9,"META-BOSEM BLUE FOR MEN ULTRA Cologne, Eau de ...","[0.033279482, 0.044996582, -0.046284977, 0.016...",0.933563


In [23]:
test_results.to_pandas()['title'].tolist()

['Michael Malul OCEAN NOIR 3.4 EAU DE PARFUM NEWEST MEN',
 "Hermes Terre D'hermes Pure Perfume for Men, Limited Edition, 2.5 Ounce",
 "Kenzo Jeu d'Amour l'Elixir Eau de Parfum Intense 2.5oz (75ml) Spray",
 "Gianni Versace Dreamer Men's 3.3-ounce Eau de Toilette Spray",
 'META-BOSEM BLUE FOR MEN ULTRA Cologne, Eau de Toilette Spray for Men, Wonderful Gift, Oriental Fougere, all Skin Types, a Classic Bottle, 3.4 Fluid Ounce']

### Sample lexical search

In [26]:
table.create_fts_index("title", replace=True)

In [27]:
test_results = table.search(test_query, query_type="fts").limit(5)

In [28]:
test_results.to_pandas()

,parent_asin,title,vector,_score
0,B000NI2MDC,Invicta Men's Watch 3480,"[-0.00073714944, 0.05022926, 0.0060243495, -0....",9.485362
1,B01M23ZZS9,INVINCIBLE By Sandora For Men EDP Perfume Frag...,"[0.022104155, 0.024795096, -0.035534833, -0.01...",7.716799
2,B00D7EWILA,"Hermes Terre D'hermes Pure Perfume for Men, Li...","[0.02430687, 0.032649007, -0.037101142, -0.013...",7.484225
3,B071KS2YDK,"Betray For Men-au De Toilette Spray Perfume, F...","[0.022353029, 0.03355059, -0.030477576, 0.0034...",5.154137
4,B0BHTGX1Z4,BARBER MARMARA Explosion Fire Limited Edition ...,"[0.03257741, 0.0895591, -0.07210774, -0.011252...",4.759031


### Sample hybrid search

In [30]:
test_results = table.search(test_query, query_type="hybrid").limit(5)

In [31]:
test_results.to_pandas()

,parent_asin,title,vector,_relevance_score
0,B00D7EWILA,"Hermes Terre D'hermes Pure Perfume for Men, Li...","[0.02430687, 0.032649007, -0.037101142, -0.013...",0.032002
1,B09HL8SVMJ,Michael Malul OCEAN NOIR 3.4 EAU DE PARFUM NEW...,"[0.030007359, 0.02743247, -0.037385404, -0.022...",0.016393
2,B000NI2MDC,Invicta Men's Watch 3480,"[-0.00073714944, 0.05022926, 0.0060243495, -0....",0.016393
3,B01M23ZZS9,INVINCIBLE By Sandora For Men EDP Perfume Frag...,"[0.022104155, 0.024795096, -0.035534833, -0.01...",0.016129
4,B01HQFOWOQ,Kenzo Jeu d'Amour l'Elixir Eau de Parfum Inten...,"[0.001010046, 0.03055501, -0.07820173, -0.0382...",0.015873
